# run_colmap — Colab GPU version

GPU counterpart of `scripts/colmap/run_colmap.py`. It runs the same COLMAP sparse reconstruction and prints the same benchmark report, but does SIFT extraction and matching on a CUDA GPU. Feature extraction is ~98% of the wall clock on the Mac (2025 s of 2062 s for `yunnan_rock_1` at full resolution), and that is the part the GPU speeds up.

Same settings as the script: 4000 SIFT keypoints, `SIMPLE_RADIAL`, one shared camera, exhaustive matching. The `benchmark.json` it writes has the same format, so results drop straight into `results/colmap_out/`.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (or better).
2. Upload the photo folders to Google Drive, laid out like the repo:
   `MyDrive/AI6121_Project/photos/<scene>/img_001.jpg …`
3. Run the cells top to bottom. Results land in `MyDrive/AI6121_Project/results/colmap_out/<scene>/`, the same layout as the local `results/colmap_out/`. Download those folders into the repo.

**Output per scene** (same as `run_colmap.py`): `benchmark.json`, `database.db`, `sparse/0_raw/` (COLMAP's own gauge + `txt/`; quote numbers from here), `sparse/0_yup/` (rotated so gravity = +Y; viewing only), `<scene>_yup2.ply`, `<scene>_with_cameras.ply` and `preview.jpg`. Set `MAKE_YUP = False` to keep just COLMAP's `sparse/0`.

**Comparing numbers:** GPU SIFT and CPU SIFT find slightly different keypoints, so a GPU run and a CPU run on the same photos won't match exactly. For the normal-vs-lower comparison, run *every* set you compare here on the GPU, including `yunnan_rock_1` if it's part of the comparison. Don't mix these numbers with the local CPU benchmark.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Install CUDA-enabled pycolmap

`pycolmap-cuda12` is the CUDA 12 build of the same pycolmap 4.2.0 the Mac uses. The plain `pycolmap` wheel is CPU-only, so it's removed first in case something pulled it in.

In [ ]:
%pip uninstall -y -q pycolmap
%pip install -q pycolmap-cuda12==4.2.0

import pycolmap
print("pycolmap", pycolmap.__version__, "| has_cuda =", pycolmap.has_cuda)
assert pycolmap.has_cuda, (
    "This pycolmap build has no CUDA. Check that the runtime has a GPU "
    "(cell 1), then Runtime -> Restart session and run from cell 2 again.")

## 3. Settings

`SCENES` run **one at a time, in order**. Each scene's results are copied to Drive as soon as it finishes, so a Colab disconnect only loses the scene in progress.

A scene whose results already exist on Drive is **skipped**, so an earlier run is never overwritten. Set `FORCE = True` to redo it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT  = "/content/drive/MyDrive/AI6121_Project"
PHOTOS_DIR     = f"{DRIVE_PROJECT}/photos"
OUT_DIR        = f"{DRIVE_PROJECT}/results/colmap_out"

SCENES         = ["yunnan_rock_2", "yunnan_rock_2_lower_height"]

MAX_IMAGE_SIZE = 4032          # full resolution for the iPhone 13 Pro's 3024x4032
MATCHER        = "exhaustive"  # or "sequential"
VOCAB_TREE     = None          # sequential only: path to vocab_tree_flickr100K_*.bin
MULTI_CAMERA   = False
FORCE          = False         # True = redo scenes that already have results on Drive
VERBOSE        = False         # True = show COLMAP's own progress logging
MAKE_YUP       = True          # build sparse/0_yup, the PLYs and preview.jpg (make_yup.py)

LOCAL_PHOTOS    = "/content/photos"      # working copies on the VM's local disk:
LOCAL_WORKSPACE = "/content/colmap_out"  # SQLite on the Drive mount is slow and unreliable

import os
for s in SCENES:
    p = f"{PHOTOS_DIR}/{s}"
    print(f"{s:32} {'found' if os.path.isdir(p) else 'MISSING  <- upload to ' + p}")

## 4. Benchmark code

`natural_key`, `list_images`, `Stopwatch`, `database_stats`, `read_text_model`, `find_models` and `report` are copied **verbatim** from `run_colmap.py`, so the numbers are computed the same way. `yup_transform`, the PLY writers, `write_preview` and `process` are copied verbatim from `make_yup.py`. The only new part is the backend, which pins extraction and matching to the GPU.

In [ ]:
import json, os, shutil, sqlite3, statistics, struct, sys, time
import numpy as np
import pycolmap

N_FEATURES = 4000
CAMERA_MODEL = "SIMPLE_RADIAL"
SEQ_OVERLAP = 10
GOOD_REGISTERED = 0.95
WEAK_REGISTERED = 0.80
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
MAX_PAIR_ID = 2147483647


def natural_key(name):
    """Sort sfm2.jpg before sfm10.jpg, not after."""
    parts, digits = [], ""
    for ch in name:
        if ch.isdigit():
            digits += ch
        else:
            if digits:
                parts.append(int(digits))
                digits = ""
            parts.append(ch.lower())
    if digits:
        parts.append(int(digits))
    return parts


def list_images(folder):
    return sorted([f for f in os.listdir(folder) if f.endswith(IMAGE_EXTS)],
                  key=natural_key)


def fmt_time(seconds):
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(seconds, 60)
    return f"{int(m)}m {s:04.1f}s"


class Stopwatch:
    """Times named stages and keeps them in insertion order."""

    def __init__(self):
        self.stages = {}

    def run(self, name, fn):
        print(f"  [{name}] running...", flush=True)
        t0 = time.perf_counter()
        result = fn()
        elapsed = time.perf_counter() - t0
        self.stages[name] = elapsed
        print(f"  [{name}] {fmt_time(elapsed)}", flush=True)
        return result

    @property
    def total(self):
        return sum(self.stages.values())


def database_stats(db_path):
    """Feature counts and two-view geometry inliers, straight from the DB.

    two_view_geometries.rows is the number of RANSAC-verified inlier matches
    for that pair - the same quantity check_photos.py calls `inliers`.
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    names = dict(cur.execute("SELECT image_id, name FROM images"))
    kps = dict(cur.execute("SELECT image_id, rows FROM keypoints"))

    pairs = []
    for pair_id, rows, config in cur.execute(
            "SELECT pair_id, rows, config FROM two_view_geometries"):
        if not rows:
            continue
        id2 = pair_id % MAX_PAIR_ID
        id1 = (pair_id - id2) // MAX_PAIR_ID
        pairs.append({
            "image1": names.get(id1, str(id1)),
            "image2": names.get(id2, str(id2)),
            "inliers": rows,
            "config": config,
        })
    con.close()

    feat = sorted(kps.values())
    inl = sorted(p["inliers"] for p in pairs)
    return {
        "features_per_image_median": int(statistics.median(feat)) if feat else 0,
        "features_per_image_min": feat[0] if feat else 0,
        "verified_pairs": len(pairs),
        "inliers_median": int(statistics.median(inl)) if inl else 0,
        "inliers_min": inl[0] if inl else 0,
        "inliers_max": inl[-1] if inl else 0,
        "pairs": pairs,
    }


def read_text_model(txt_dir):
    """Parse cameras.txt / images.txt / points3D.txt into benchmark numbers."""
    images, points, cameras = {}, {}, {}

    with open(os.path.join(txt_dir, "cameras.txt")) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            f = line.split()
            cameras[int(f[0])] = {
                "model": f[1],
                "width": int(f[2]),
                "height": int(f[3]),
                "params": [float(x) for x in f[4:]],
            }

    with open(os.path.join(txt_dir, "images.txt")) as fh:
        lines = [ln for ln in fh if not ln.startswith("#")]
    i = 0
    while i < len(lines):
        header = lines[i].split()
        i += 1
        if not header:
            continue
        pts = lines[i].split() if i < len(lines) else []
        i += 1
        # POINTS2D triples; a POINT3D_ID of -1 means the keypoint is unused.
        observed = sum(1 for k in range(2, len(pts), 3) if pts[k] != "-1")
        images[int(header[0])] = {
            "name": header[9],
            "camera_id": int(header[8]),
            "observations": observed,
            "keypoints": len(pts) // 3,
        }

    errors, track_lengths = [], []
    with open(os.path.join(txt_dir, "points3D.txt")) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            f = line.split()
            error = float(f[7])
            track = (len(f) - 8) // 2
            errors.append(error)
            track_lengths.append(track)
            points[int(f[0])] = (error, track)

    n_obs = sum(track_lengths)
    # COLMAP's model_analyzer weights each point's error by its track length.
    mean_reproj = (sum(e * t for e, t in zip(errors, track_lengths)) / n_obs
                   if n_obs else 0.0)

    obs_per_image = [im["observations"] for im in images.values()]
    return {
        "registered_images": len(images),
        "points3D": len(points),
        "observations": n_obs,
        "mean_track_length": n_obs / len(points) if points else 0.0,
        "mean_observations_per_image": (n_obs / len(images)) if images else 0.0,
        "min_observations_per_image": min(obs_per_image) if obs_per_image else 0,
        "mean_reprojection_error_px": mean_reproj,
        "cameras": cameras,
        "image_names": sorted((im["name"] for im in images.values()),
                              key=natural_key),
    }


def find_models(sparse_dir):
    if not os.path.isdir(sparse_dir):
        return []
    out = []
    for entry in sorted(os.listdir(sparse_dir), key=natural_key):
        if entry.endswith("_yup"):      # viewing copy of another model, not a model
            continue
        path = os.path.join(sparse_dir, entry)
        if os.path.isdir(path) and (
                os.path.exists(os.path.join(path, "images.bin"))
                or os.path.exists(os.path.join(path, "images.txt"))):
            out.append(path)
    return out


def report(folder, names, models, stats, watch, backend, settings, out_json):
    backend_name = f"{backend.name} ({backend.device} SIFT)"
    n_total = len(names)
    best = max(models, key=lambda m: m["registered_images"])
    frac = best["registered_images"] / n_total

    print("\n" + "=" * 68)
    print(f"COLMAP BENCHMARK  -  {os.path.basename(os.path.abspath(folder))}")
    print("=" * 68)

    print(f"\nbackend                     : {backend_name}")
    print(f"matcher                     : {settings['matcher']}")
    print(f"max image size / keypoints  : {settings['max_image_size']} px"
          f" / {N_FEATURES}")
    print(f"input images                : {n_total}")
    print(f"models reconstructed        : {len(models)}"
          + ("  (should be 1 - the set split apart)" if len(models) > 1 else ""))

    print("\n-- largest model " + "-" * 51)
    print(f"registered images           : {best['registered_images']} / "
          f"{n_total}   ({frac * 100:.0f}%)")
    print(f"sparse 3D points            : {best['points3D']}")
    print(f"observations (2D-3D links)  : {best['observations']}")
    print(f"mean track length           : {best['mean_track_length']:.2f}"
          "   (images seeing each point)")
    print(f"mean observations / image   : "
          f"{best['mean_observations_per_image']:.0f}")
    print(f"min observations / image    : {best['min_observations_per_image']}")
    print(f"mean reprojection error     : "
          f"{best['mean_reprojection_error_px']:.3f} px")
    for cam in best["cameras"].values():
        print(f"camera                      : {cam['model']} "
              f"{cam['width']}x{cam['height']}  f={cam['params'][0]:.1f} px")
        break

    print("\n-- two-view geometry (COLMAP's own RANSAC) " + "-" * 25)
    print(f"descriptors / image (median): "
          f"{stats['features_per_image_median']}"
          f"   (from a {N_FEATURES}-keypoint cap)")
    print(f"verified pairs              : {stats['verified_pairs']}")
    print(f"inliers per pair            : median "
          f"{stats['inliers_median']}, min {stats['inliers_min']}, "
          f"max {stats['inliers_max']}")

    print("\n-- wall clock " + "-" * 54)
    if watch.stages:
        for stage, secs in watch.stages.items():
            print(f"{stage:<28}: {fmt_time(secs)}")
        print(f"{'TOTAL':<28}: {fmt_time(watch.total)}"
              f"   ({watch.total / n_total:.1f}s per image)")
    else:
        print("reported from an existing workspace - nothing was recomputed")

    missing = sorted(set(names) - set(best["image_names"]), key=natural_key)
    if missing:
        print(f"\ndropped images ({len(missing)}):")
        print("   " + ", ".join(missing))

    print()
    if len(models) > 1:
        print("VERDICT: SPLIT. COLMAP built more than one disconnected model.")
        print("   The photo chain breaks somewhere - the dropped images above")
        print("   are the place to look. Reshoot the gap and rerun.")
    elif frac >= GOOD_REGISTERED:
        print("VERDICT: GOOD. One connected model, nearly every photo placed.")
    elif frac >= WEAK_REGISTERED:
        print("VERDICT: PARTIAL. One model, but some photos never registered.")
        print("   Usable, though the dropped views contribute nothing.")
    else:
        print("VERDICT: POOR. Most photos failed to register.")
        print("   Treat this set as not reconstructable without more coverage.")

    summary = {
        "folder": os.path.abspath(folder),
        "backend": backend_name,
        "settings": settings,
        "input_images": n_total,
        "models_reconstructed": len(models),
        "largest_model": {k: v for k, v in best.items()
                          if k not in ("cameras", "image_names")},
        "registered_fraction": frac,
        "dropped_images": missing,
        "two_view_geometry": {k: v for k, v in stats.items() if k != "pairs"},
        "timing_seconds": dict(watch.stages, TOTAL=watch.total),
    }
    with open(out_json, "w") as fh:
        json.dump(summary, fh, indent=2)
    print(f"\nmachine-readable summary -> {out_json}")
    print()


def yup_transform(rec):
    """Return (R, c0): x_yup = R @ (x_raw - c0)."""
    centres = np.array([im.projection_center() for im in rec.images.values()])
    c0 = centres.mean(0)
    n = np.linalg.svd(centres - c0)[2][2]
    pts = np.array([p.xyz for p in rec.points3D.values()])
    if np.median((pts - c0) @ n) < 0:
        n = -n
    y = np.array([0.0, 1.0, 0.0])
    v, c = np.cross(n, y), n @ y
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    R = np.eye(3) + K + K @ K / (1 + c)
    return R, c0


def write_ply_binary(path, xyz, rgb):
    header = ("ply\nformat binary_little_endian 1.0\n"
              f"element vertex {len(xyz)}\n"
              "property float x\nproperty float y\nproperty float z\n"
              "property uchar red\nproperty uchar green\nproperty uchar blue\n"
              "end_header\n").encode()
    with open(path, "wb") as fh:
        fh.write(header)
        for p, c in zip(xyz, rgb):
            fh.write(struct.pack("<fffBBB", *p, *c))


def write_ply_ascii(path, xyz, rgb):
    with open(path, "w") as fh:
        fh.write("ply\nformat ascii 1.0\n"
                 f"element vertex {len(xyz)}\n"
                 "property float x\nproperty float y\nproperty float z\n"
                 "property uchar red\nproperty uchar green\nproperty uchar blue\n"
                 "end_header\n")
        for p, c in zip(xyz, rgb):
            fh.write(f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f} {c[0]} {c[1]} {c[2]}\n")


def write_preview(path, xyz, rgb, cams, panel=600):
    """Two 600x600 panels on a dark ground: top-down (x-z) and side (x-y, +Y down)."""
    import cv2

    # frame both views on the camera ring; far background points fall outside
    r = np.abs(cams[:, [0, 2]]).max() * 1.15
    scale = (panel / 2 - 20) / r
    img = np.full((panel, 2 * panel, 3), 17, np.uint8)
    order = np.argsort(-xyz[:, 1])              # far-below first, so upper points draw on top

    def draw(ox, u, v, pts, cols, radius):
        px = np.round(ox + panel / 2 + u * scale).astype(int)
        py = np.round(panel / 2 + v * scale).astype(int)
        ok = (px >= ox) & (px < ox + panel) & (py >= 30) & (py < panel)
        for x, y, c in zip(px[ok], py[ok], cols[ok]):
            cv2.circle(img, (int(x), int(y)), radius, tuple(int(k) for k in c[::-1]), -1)

    p, col = xyz[order], rgb[order]
    red = np.tile([255, 0, 0], (len(cams), 1))
    draw(0, p[:, 0], p[:, 2], p, col, 1)                        # top-down: x right, z down
    draw(0, cams[:, 0], cams[:, 2], cams, red, 4)
    y_mid = (cams[:, 1].mean() + np.percentile(xyz[:, 1], 90)) / 2
    draw(panel, p[:, 0], p[:, 1] - y_mid, p, col, 1)            # side: +Y down the page
    draw(panel, cams[:, 0], cams[:, 1] - y_mid, cams, red, 4)
    for ox, text in ((0, f"TOP-DOWN (x-z): {len(cams)} cameras"),
                     (panel, "SIDE (x-y, +Y down)")):
        cv2.putText(img, text, (ox + 10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (220, 220, 220), 1, cv2.LINE_AA)
    cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, 90])


def process(workspace):
    scene = os.path.basename(os.path.normpath(workspace))
    sparse = os.path.join(workspace, "sparse")
    raw_dir = os.path.join(sparse, "0_raw")
    yup_dir = os.path.join(sparse, "0_yup")

    if os.path.isdir(yup_dir):
        raise RuntimeError(f"{yup_dir} already exists - remove it first to rebuild.")
    if not os.path.isdir(raw_dir):
        models = [d for d in os.listdir(sparse)
                  if os.path.exists(os.path.join(sparse, d, "images.bin"))]
        if models != ["0"]:
            raise RuntimeError(f"{sparse}: expected exactly one model '0', found "
                               f"{models}. Pick one by hand before running this.")
        os.rename(os.path.join(sparse, "0"), raw_dir)
        print(f"  sparse/0 -> sparse/0_raw")

    rec = pycolmap.Reconstruction(raw_dir)
    R, c0 = yup_transform(rec)
    rec.transform(pycolmap.Sim3d(1.0, pycolmap.Rotation3d(R), -R @ c0))
    os.makedirs(yup_dir)
    rec.write(yup_dir)
    rec = pycolmap.Reconstruction(yup_dir)      # PLYs list cameras in on-disk order

    pts = list(rec.points3D.values())
    xyz = np.array([p.xyz for p in pts])        # float64; the ascii PLY rounds from these
    rgb = np.array([p.color for p in pts], dtype=np.uint8)
    cams = np.array([im.projection_center() for im in rec.images.values()])

    write_ply_binary(os.path.join(workspace, f"{scene}_yup2.ply"),
                     xyz.astype(np.float32), rgb)
    write_ply_ascii(os.path.join(workspace, f"{scene}_with_cameras.ply"),
                    np.vstack([xyz, cams]),
                    np.vstack([rgb, np.tile([255, 0, 0], (len(cams), 1))]).astype(int))
    write_preview(os.path.join(workspace, "preview.jpg"), xyz, rgb, cams)

    heights = cams[:, 1]
    print(f"  0_yup written: {len(rec.images)} cameras, {len(pts)} points; "
          f"camera height spread {heights.std():.3f} (should be small for a level ring)")
    print(f"  {scene}_yup2.ply, {scene}_with_cameras.ply, preview.jpg")


class GpuBackend:
    """pycolmap with SIFT extraction and matching pinned to CUDA."""

    def __init__(self, max_image_size, verbose):
        self.pc = pycolmap
        self.name = f"pycolmap-cuda12 {pycolmap.__version__}"
        self.device = "GPU"
        self.max_image_size = max_image_size
        if not verbose:
            pycolmap.logging.minloglevel = pycolmap.logging.Level.WARNING

    def extract(self, db, image_dir, names, single_camera):
        pc = self.pc
        reader = pc.ImageReaderOptions()
        reader.camera_model = CAMERA_MODEL
        opts = pc.FeatureExtractionOptions()
        opts.sift.max_num_features = N_FEATURES
        opts.max_image_size = self.max_image_size
        opts.use_gpu = True
        # image_names keeps stray files (e.g. .scene) out of the reader
        pc.extract_features(db, image_dir, image_names=names,
                            camera_mode=(pc.CameraMode.SINGLE if single_camera
                                         else pc.CameraMode.AUTO),
                            reader_options=reader, extraction_options=opts,
                            device=pc.Device.cuda)

    def match(self, db, mode, vocab_tree):
        pc = self.pc
        mopts = pc.FeatureMatchingOptions()
        mopts.use_gpu = True
        if mode != "sequential":
            pc.match_exhaustive(db, matching_options=mopts, device=pc.Device.cuda)
            return
        pairing = pc.SequentialPairingOptions()
        pairing.overlap = SEQ_OVERLAP
        if vocab_tree:
            pairing.loop_detection = True
            pairing.vocab_tree_path = vocab_tree
        pc.match_sequential(db, matching_options=mopts, pairing_options=pairing,
                            device=pc.Device.cuda)

    def map(self, db, image_dir, sparse_dir):
        os.makedirs(sparse_dir, exist_ok=True)
        recs = self.pc.incremental_mapping(db, image_dir, sparse_dir)
        if recs and not find_models(sparse_dir):
            items = recs.items() if hasattr(recs, "items") else enumerate(recs)
            for idx, rec in items:
                out = os.path.join(sparse_dir, str(idx))
                os.makedirs(out, exist_ok=True)
                rec.write(out)

    def to_text(self, model_dir, txt_dir):
        os.makedirs(txt_dir, exist_ok=True)
        self.pc.Reconstruction(model_dir).write_text(txt_dir)


def run_scene(scene, backend):
    """Same flow as run_colmap.py main(), for one scene. Returns the summary dict."""
    drive_out = os.path.join(OUT_DIR, scene)
    if os.path.exists(os.path.join(drive_out, "benchmark.json")) and not FORCE:
        print(f"SKIP {scene}: results already on Drive at {drive_out} (FORCE = True to redo)")
        return json.load(open(os.path.join(drive_out, "benchmark.json")))

    # copy only the image files to local disk
    folder = os.path.join(LOCAL_PHOTOS, scene)
    os.makedirs(folder, exist_ok=True)
    names = list_images(os.path.join(PHOTOS_DIR, scene))
    if len(names) < 3:
        raise SystemExit(f"{scene}: need at least 3 images, found {len(names)}")
    for n in names:
        dst = os.path.join(folder, n)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(PHOTOS_DIR, scene, n), dst)

    workspace = os.path.join(LOCAL_WORKSPACE, scene)
    if os.path.exists(workspace):
        shutil.rmtree(workspace)          # local scratch only; Drive copy is untouched
    os.makedirs(workspace)
    db = os.path.join(workspace, "database.db")
    sparse = os.path.join(workspace, "sparse")

    print(f"\n{len(names)} images in {scene}")
    print(f"backend   : {backend.name}")
    print(f"workspace : {workspace}")
    print(f"matcher   : {MATCHER}{' + loop detection' if VOCAB_TREE else ''}\n")

    watch = Stopwatch()
    watch.run("features", lambda: backend.extract(db, folder, names,
                                                   single_camera=not MULTI_CAMERA))
    watch.run("matching", lambda: backend.match(db, MATCHER, VOCAB_TREE))
    watch.run("mapping",  lambda: backend.map(db, folder, sparse))

    model_dirs = find_models(sparse)
    if not model_dirs:
        print(f"\n{scene}: COLMAP produced no reconstruction at all.")
        return None
    models = []
    for model_dir in model_dirs:
        txt_dir = os.path.join(model_dir, "txt")
        if not os.path.exists(os.path.join(txt_dir, "images.txt")):
            backend.to_text(model_dir, txt_dir)
        models.append(read_text_model(txt_dir))

    stats = database_stats(db)
    settings = {
        "matcher": MATCHER,
        "max_image_size": MAX_IMAGE_SIZE,
        "max_num_features": N_FEATURES,
        "camera_model": CAMERA_MODEL,
        "single_camera": not MULTI_CAMERA,
        "loop_detection": bool(VOCAB_TREE),
    }
    out_json = os.path.join(workspace, "benchmark.json")
    report(folder, names, models, stats, watch, backend, settings, out_json)
    if MAKE_YUP:
        print("  [yup] building the gravity = +Y viewing copy...")
        try:
            process(workspace)
        except RuntimeError as e:
            print(f"  [yup] skipped: {e}")

    # save to Drive immediately, so a disconnect later doesn't lose this scene
    if os.path.exists(drive_out):
        shutil.rmtree(drive_out)
    shutil.copytree(workspace, drive_out)
    print(f"saved -> {drive_out}")
    return json.load(open(out_json))

print("ready")

## 5. Run, one scene at a time

In [ ]:
backend = GpuBackend(MAX_IMAGE_SIZE, VERBOSE)
results = {}
for scene in SCENES:
    print("#" * 68 + f"\n# {scene}\n" + "#" * 68)
    results[scene] = run_scene(scene, backend)

## 6. Side-by-side summary

**Different image counts:** point counts and observation totals grow with the number of images, so they aren't a fair comparison between sets of different size. Registered fraction, mean track length, observations per image and reprojection error are fair. For a strict comparison, subsample the sets to the same count first.

In [ ]:
rows = [
    ("input images",            lambda s: s["input_images"]),
    ("models reconstructed",    lambda s: s["models_reconstructed"]),
    ("registered",              lambda s: f'{s["largest_model"]["registered_images"]} ({s["registered_fraction"]*100:.0f}%)'),
    ("sparse 3D points",        lambda s: s["largest_model"]["points3D"]),
    ("mean track length",       lambda s: f'{s["largest_model"]["mean_track_length"]:.2f}'),
    ("mean obs / image",        lambda s: f'{s["largest_model"]["mean_observations_per_image"]:.0f}'),
    ("mean reproj error (px)",  lambda s: f'{s["largest_model"]["mean_reprojection_error_px"]:.3f}'),
    ("verified pairs",          lambda s: s["two_view_geometry"]["verified_pairs"]),
    ("inliers / pair (median)", lambda s: s["two_view_geometry"]["inliers_median"]),
    ("dropped images",          lambda s: len(s["dropped_images"])),
    ("total time",              lambda s: fmt_time(s["timing_seconds"]["TOTAL"])),
]
done = {k: v for k, v in results.items() if v}
print(f"{'':26}" + "".join(f"{k[:26]:>28}" for k in done))
for label, fn in rows:
    print(f"{label:26}" + "".join(f"{str(fn(s)):>28}" for s in done.values()))
missing = [k for k, v in results.items() if not v]
if missing:
    print("\nno reconstruction:", ", ".join(missing))